In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [5]:
len(documents)

72

In [6]:
from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

index.fit(documents)

In [7]:
question = 'How does the agentic loop keep calling the model until it stops?'
search_result = index.search(
    question, 
    num_results=5
)

search_result[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

In [15]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
""".strip()

PROMPT_TEMPLATE = """
QUESTION: {question} 

CONTEXT:
{context}
""".strip()

In [21]:
class RAGBase:
    def __init__(
            self,
            index,
            llm_client,
            instructions = INSTRUCTIONS,
            prompt_template = PROMPT_TEMPLATE,
            model = 'gpt-5.4-mini'
        ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.prompt_template = prompt_template
        self.model = model 
    
    def search(self, question):
        return self.index.search(
            question, 
            num_results = 5
        )
    
    def build_context(self, search_result):
        lines = []
        for doc in search_result:
            lines.append('filename:' + doc['filename'])
            lines.append('content: ' + doc['content'])
            lines.append('')
        return '\n'.join(lines).strip()
    
    def build_prompt(self, question, search_results):
        context = self.build_context(search_results)
        prompt = self.prompt_template.format(
            question=question,
            context=context
        )
        return prompt.strip()

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response.output_text, response.usage
    
    def rag(self,query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer, usage = self.llm(prompt)
        return answer, usage

In [22]:
assistant = RAGBase(
    index = index,
    llm_client=openai_client
)

query = 'How does the agentic loop keep calling the model until it stops?'
answer, usage = assistant.rag(query)


In [23]:
usage.input_tokens

7131

In [20]:
answer.output_text

AttributeError: 'str' object has no attribute 'output_text'